<a href="https://colab.research.google.com/github/DaniloDuque/neural-network/blob/main/src/tp3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP3 — Redes Neuronales

Este notebook está organizado en tres secciones principales:

1. **Two Moons** — clasificación binaria no lineal para verificar la capacidad del MLP.
2. **Clasificador en R²** — 4 configuraciones de MLP (M=20/2, con/sin momentum) sobre datos separables y no separables.
3. **ACRIMA** — clasificación de imágenes de fondo de ojo (glaucoma vs. no glaucoma).

**Modelo:** Perceptrón Multicapa con descenso de gradiente con momentum (Bishop, 2006).  

## Configuración del entorno

Se clona el repositorio si se ejecuta en Google Colab, se agrega `src/` al path de Python y se configuran las credenciales de Kaggle necesarias para la Sección 3.

In [ ]:
## @brief Configura el entorno de ejecución (Colab o local).
#  Clona el repositorio si se ejecuta en Google Colab y ajusta el directorio de trabajo.
import os, sys
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
    if not os.path.exists('neural-network'):
        os.system('git clone https://github.com/DaniloDuque/neural-network.git')
    if os.path.basename(os.getcwd()) != 'neural-network':
        os.chdir('neural-network')
    os.system('git pull')

sys.path.insert(0, os.path.abspath('src'))

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print('Credenciales cargadas desde Colab Secrets.')
    except Exception:
        # Corriendo desde VS Code con extensión Colab — usar .env
        load_dotenv()
        print('Credenciales cargadas desde .env (extensión VS Code).')
else:
    kaggle_cfg = os.path.expanduser('~/.config/kaggle/kaggle.json')
    kagglehub_cache = os.path.expanduser('~/.cache/kagglehub/datasets/toaharahmanratul/acrima-dataset')
    if os.path.exists(kaggle_cfg):
        print(f'Credenciales encontradas en {kaggle_cfg}')
    elif os.path.exists(kagglehub_cache):
        print('Dataset ACRIMA ya en caché local — no se requieren credenciales.')
    else:
        print('ADVERTENCIA: no se encontró kaggle.json')

## Dependencias

Se instalan las dependencias necesarias para ejecutar el notebook.

In [ ]:
## @brief Instala dependencias adicionales no incluidas en Colab.
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'kagglehub==0.3.13', 'Pillow', '-q'],
               check=True)

## Dependencias e importaciones

Se importan todos los módulos del proyecto y se detecta el dispositivo de cómputo.

In [ ]:
## @brief Importa todos los módulos del proyecto y configura el entorno.
#  Detecta el dispositivo disponible (CUDA o CPU) para pasarlo a las funciones de biblioteca.
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
from PIL import Image
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import kagglehub

from multilayer_perceptron import MultilayerPerceptron
from data_generator        import generate_data, generate_moons_data
from data_classifier       import train_all_configs
from visualization         import (plot_datasets, plot_decision_surface_moons, plot_error_matrix,
                                   print_convergence_table, convergence_dataframe)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')

FIGURES_DIR = Path('..') / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)
RANDOM_SEED=42
import pandas as pd
from xor_classifier        import (CONFIGS, CONVERGENCE_THRESHOLD,
                                    train_xor_configuration, convergence_epoch)


---
# Sección 1 — Desarrollo de Multilayer_Perceptron

Para ver los detalles visitar la documentación

---
# Sección 2 — Clasificación de Función XOR

Se entrena un MLP con arquitectura **[D, M, K] = [2, 2, 1]** sobre la tabla de verdad XOR.
El problema XOR no es linealmente separable: ningún hiperplano en R² puede dividir correctamente
las cuatro muestras. La capa oculta introduce una representación no lineal que permite al modelo
aprender la frontera de decisión correcta.

**Arquitectura:**
- D = 2 neuronas de entrada (x₁, x₂)
- M = 2 neuronas ocultas (representación no lineal)
- K = 1 neurona de salida (predicción binaria)

El bias es manejado internamente por `MultilayerPerceptron`.

In [ ]:
## @brief Define el dataset XOR.

X_xor = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
T_xor = torch.tensor([[0.], [1.], [1.], [0.]])
print('Dataset XOR:')
print(torch.cat([X_xor, T_xor], dim=1))

## 2.1 — Entrenamiento con múltiples configuraciones

In [ ]:
## @brief Entrena el MLP XOR con todas las configuraciones de hiperparámetros.
torch.manual_seed(RANDOM_SEED)

xor_results = [train_xor_configuration(alpha, gamma, num_epochs=40000) for alpha, gamma in CONFIGS]

for r in xor_results:
    print(f"α={r['alpha']:.2f}  γ={r['gamma']:.1f}  →  error final: {r['final_error']:.6f}")

## 2.2 — Curvas de aprendizaje

In [ ]:
## @brief Genera y guarda las curvas de aprendizaje MSE vs. iteraciones.
fig, ax = plt.subplots(figsize=(8, 5))

maxvalues = [max(r['errors']) for r in xor_results]

max_error = max(maxvalues)
for r in xor_results:
    ax.plot(r['errors'], label=f"α={r['alpha']:.2f} γ={r['gamma']:.1f}")
    # n = 0
    # for err in r['errors']:
    #     if(n%500==0):
    #         print(f"α={r['alpha']:.2f} γ={r['gamma']:.1f}  →  error: {err:.6f}  (época {n})")
    #     n+=1


ax.set_xlabel('Época')
ax.set_ylabel('Error MSE')
ax.set_title('XOR — Curvas de aprendizaje')
ax.legend()
ax.set_yscale('linear')
ax.set_ylim(1e-4, max_error)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xor_learning_curves.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'xor_learning_curves.png', bbox_inches='tight')
plt.show()

## 2.3 — Tabla de resultados

In [ ]:
## @brief Construye la tabla comparativa de hiperparámetros y error final.

df_xor = pd.DataFrame([
    {
        'α': r['alpha'],
        'γ': r['gamma'],
        'Error Final': r['final_error'],
        'Época Convergencia': convergence_epoch(r['errors'], CONVERGENCE_THRESHOLD),
    }
    for r in xor_results
])
df_xor

## 2.4 — Validación final (mejor configuración)

In [ ]:
## @brief Evalúa la mejor configuración y muestra las predicciones finales.
best = min(xor_results, key=lambda r: r['final_error'])
print(f"Mejor configuración: α={best['alpha']:.2f}  γ={best['gamma']:.1f}")
print(f"Error final: {best['final_error']:.6f}")
print(f"Época de convergencia: {convergence_epoch(best['errors'], CONVERGENCE_THRESHOLD)}")

preds = best['mlp'].predict(X_xor)

df_val = pd.DataFrame({
    'x₁': X_xor[:, 0].int().tolist(),
    'x₂': X_xor[:, 1].int().tolist(),
    'Esperado': T_xor.squeeze().int().tolist(),
    'Predicción': preds.squeeze().tolist(),
})
df_val

---
# Sección 3 — Clasificador de datos en R²

Se generan dos datasets (separable / no-separable) y se entrenan 4 configuraciones de MLP:

| Config | M  | γ   | Descripción               |
|--------|----|-----|---------------------------|
| A      | 20 | 0.9 | 20 neuronas, con momentum |
| B      | 20 | 0.0 | 20 neuronas, sin momentum |
| C      | 2  | 0.9 | 2 neuronas, con momentum  |
| D      | 2  | 0.0 | 2 neuronas, sin momentum  |

Partición 80% entrenamiento / 20% validación. α = 0.01 fijo para todas las configuraciones.

In [ ]:
## @brief Define hiperparámetros globales para la sección 2.
NUM_EPOCHS = 50

## 3.1 — Generación de datos

In [ ]:
## @brief Genera los datasets separable y no-separable con split 80/20.
X_train_s_b, X_val_s_b, T_train_s_b, T_val_s_b = generate_data(
    separable=True,  n_samples=500, test_size=0.2, random_state=42,
    device=torch.device(device)
)

X_train_ns_m, X_val_ns_m, T_train_ns_m, T_val_ns_m = generate_moons_data(
    noise=0.08, n_samples=500, test_size=0.2, random_state=42,
    device=torch.device(device)
)

#datos generado con make_blobs
print(f'Blobs: Separable    — train: {X_train_s_b.shape},  val: {X_val_s_b.shape}')

#datos generado con make_moons
print(f'Moons: No-separable — train: {X_train_ns_m.shape}, val: {X_val_ns_m.shape}')

## 3.2 — Visualización de los datos

In [ ]:
## @brief Muestra scatter de ambos datasets (train + val combinados).
X_sep_full   = torch.cat([X_train_s_b,  X_val_s_b],  dim=0).cpu()
T_sep_full   = torch.cat([T_train_s_b,  T_val_s_b],  dim=0).cpu()
X_nosep_full = torch.cat([X_train_ns_m, X_val_ns_m], dim=0).cpu()
T_nosep_full = torch.cat([T_train_ns_m, T_val_ns_m], dim=0).cpu()

fig = plot_datasets(X_sep_full, T_sep_full, X_nosep_full, T_nosep_full,
                    output_path=FIGURES_DIR / 'datasets_scatter.pdf')
plt.show()


## 3.3 — Entrenamiento de las 4 configuraciones

In [ ]:
## @brief Entrena las 4 configuraciones sobre el dataset separable.
results_sep_b = train_all_configs(
    X_train_s_b, T_train_s_b, X_val_s_b, T_val_s_b,
    num_epochs=NUM_EPOCHS
)


## @brief Entrena las 4 configuraciones sobre el dataset no-separable.
results_nosep_m = train_all_configs(
    X_train_ns_m, T_train_ns_m, X_val_ns_m, T_val_ns_m,
    num_epochs=NUM_EPOCHS
)

## 3.4 — Evolución del error

In [ ]:
## @brief Genera la matriz 2×4 de curvas de error train/val por configuración.
fig = plot_error_matrix(results_sep_b, results_nosep_m,
                        output_path=FIGURES_DIR / 'error_matrix.pdf')
plt.show()

## 3.5 — Tabla de convergencia

In [ ]:
## @brief Imprime la tabla de errores finales de validación por configuración.
print_convergence_table(results_sep_b, results_nosep_m)
convergence_dataframe(results_sep_b, results_nosep_m)

---
# Sección 4 — Clasificador de imágenes ACRIMA

Dataset: [ACRIMA en Kaggle](https://www.kaggle.com/datasets/toaharahmanratul/acrima-dataset) — imágenes de fondo de ojo etiquetadas con y sin glaucoma.

In [ ]:
from acrima_dataset import ACRIMADataset, encontrar_directorio_dataset

In [ ]:
## @brief Descarga el dataset ACRIMA desde Kaggle usando kagglehub.
#  La primera ejecución descarga (~1 GB). Las siguientes usan la caché local.
ACRIMA_PATH = kagglehub.dataset_download('toaharahmanratul/acrima-dataset')
print(f'Dataset descargado en: {ACRIMA_PATH}')

In [ ]:

train_dir = encontrar_directorio_dataset(ACRIMA_PATH)/"train"
test_dir  = Path(ACRIMA_PATH) / "test"


In [ ]:


# Crear datasets
print("Dataset de entrenamiendo:")
dataset_train_mlp  = ACRIMADataset(train_dir, img_size=(64, 64))
dataset_train_alex = ACRIMADataset(train_dir, img_size=(224, 224))

print("Dataset de pruebas:")
dataset_test_mlp  = ACRIMADataset(test_dir, img_size=(64, 64))
dataset_test_alex = ACRIMADataset(test_dir, img_size=(224, 224))

In [ ]:
# Pruebas:

# ═══ PRUEBA 1: Verificación numérica ═══════════════════════════════════════
print("PRUEBA 1 — Verificación numérica: max(img_norm) debe ser ≈ 1.0")
print(f"{'Idx':>4} | {'Max original':>13} | {'Max norm':>10} | {'Min norm':>9} | OK")
print("-"*55)

resultados = []
for i in range(min(10, len(dataset_train_mlp))):
    img_path, _ = dataset_train_mlp.samples[i]
    arr_raw = np.array(Image.open(img_path).convert("RGB").resize((64,64)), dtype=np.float32)
    max_raw = arr_raw.max()

    img_norm, _ = dataset_train_mlp[i]
    max_n = img_norm.max().item()
    min_n = img_norm.min().item()
    ok = abs(max_n - 1.0) < 1e-5 if max_raw > 0 else True
    resultados.append(ok)
    print(f"{i:>4} | {max_raw:>13.2f} | {max_n:>10.6f} | {min_n:>9.6f} | {'✓' if ok else '✗'}")

print(f"\nResultado: {'PASÓ ✓' if all(resultados) else 'FALLÓ ✗'} ({sum(resultados)}/{len(resultados)})")

# ═══ PRUEBA 2: Verificación visual ═════════════════════════════════════════
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("PRUEBA 2 — Normalización ℓ∞: Original vs Normalizada", fontsize=13, fontweight='bold')

indices = random.sample(range(len(dataset_train_mlp)), 4)
for col, idx in enumerate(indices):
    img_path, label = dataset_train_mlp.samples[idx]
    clase = dataset_train_mlp.CLASES[label]

    img_raw = np.array(Image.open(img_path).convert("RGB").resize((64, 64)))
    axes[0, col].imshow(img_raw)
    axes[0, col].set_title(f"Original — {clase}\nmax={img_raw.max()}", fontsize=9)
    axes[0, col].axis("off")

    img_norm, _ = dataset_train_mlp[idx]
    img_norm_np = img_norm.permute(1, 2, 0).numpy()
    axes[1, col].imshow(img_norm_np)
    axes[1, col].set_title(f"Normalizada ℓ∞\nmax={img_norm_np.max():.4f}", fontsize=9)
    axes[1, col].axis("off")

plt.tight_layout()
plt.savefig("prueba_normalizacion.png", dpi=120, bbox_inches='tight')
plt.show()
print("✓ Figura guardada: prueba_normalizacion.png")


In [ ]:

H, W, C = 64, 64, 3
D = H * W * C   # 12288
T = 2           # clases

CONFIGURACIONES = [
    {"nombre": "Config-1 (Pequeña)",  "D": D, "M": 256,  "T": T},
    {"nombre": "Config-2 (Media)",    "D": D, "M": 512,  "T": T},
    {"nombre": "Config-3 (Grande)",   "D": D, "M": 1024, "T": T},
]

print("Configuraciones definidas:")
print(f"{'Nombre':<25} | {'D':>7} | {'M':>6} | {'T':>4} | {'Parámetros':>12}")
print("-"*65)
for c in CONFIGURACIONES:
    params = (c['D']*c['M']+c['M']) + (c['M']*c['T']+c['T'])
    print(f"{c['nombre']:<25} | {c['D']:>7,} | {c['M']:>6,} | {c['T']:>4} | {params:>12,}")


In [ ]:
"""
Sección 4c — Calibración de hiperparámetros y entrenamiento del MLP sobre ACRIMA.

Heurísticas aplicadas:
  • M  : regla de Kolmogorov generalizada → M ≈ √(D·K) redondeado a potencia de 2.
          Con D=12288 y K=1 → √12288 ≈ 111 → se usa M=128 (potencia de 2 cercana).
          Para las otras dos configs se escalan: M=64 (underfit boundary) y M=256 (overfit risk).
  • α  : heurística de Jacobs (1988) → α = 1/(n * (1 + M/D)) ≈ 0.001 para n≈600.
          En la práctica se fija α=0.001 (estable con momentum).
  • P  : criterio de convergencia empírico → suficientes épocas para que el error
          de validación deje de decrecer. Con batch completo y α=0.001 se estima P=30.
          (El MLP es batch gradient descent, sin mini-batches.)
  • γ  : momentum estándar = 0.9 (valor popularizado por Polyak & Rumelhart).

Los tres conjuntos de M elegidos (64, 128, 256) cubren el espacio bajo/medio/alto
de capacidad del modelo, manteniendo α y P fijos para aislar el efecto de M.
"""

from torch.utils.data import DataLoader
import random

# ── Hiperparámetros calibrados ────────────────────────────────────────────────
H, W, C   = 64, 64, 3
D         = H * W * C      # 12288 — neuronas de entrada (píxeles aplanados)
K         = 1              # salida binaria (1 neurona sigmoid → 0=Normal, 1=Glaucoma)
ALPHA     = 0.001          # tasa de aprendizaje (heurística de Jacobs)
GAMMA     = 0.9            # momentum (Polyak / Rumelhart)
MAX_W     = 0.05           # inicialización de pesos: Uniform[-0.05, 0.05]
NUM_EPOCHS = 30            # épocas (convergencia empírica con batch completo)
BATCH_SZ  = None           # None = batch completo (más estable para MLP de 1 capa)

CONFIGURACIONES = [
    {"nombre": "Config-1 (M=64)",  "M": 64},
    {"nombre": "Config-2 (M=128)", "M": 128},
    {"nombre": "Config-3 (M=256)", "M": 256},
]

print("═" * 65)
print("  Hiperparámetros calibrados para ACRIMA")
print("═" * 65)
print(f"  D (entrada)  = {D:,}  (64×64×3 píxeles aplanados)")
print(f"  K (salida)   = {K}    (clasificación binaria)")
print(f"  α (lr)       = {ALPHA}  (heurística Jacobs: 1/(n·(1+M/D)))")
print(f"  γ (momentum) = {GAMMA}   (Polyak/Rumelhart estándar)")
print(f"  max_w        = {MAX_W}  (inicialización uniforme)")
print(f"  P (épocas)   = {NUM_EPOCHS}   (criterio de convergencia empírico)")
print(f"  M elegidos   = 64, 128, 256  (√D≈111, potencias de 2 cercanas)")
print("═" * 65)


# ── Preparar tensores de entrenamiento / validación ───────────────────────────
def dataset_to_tensors(dataset, device):
    """
    Aplana cada imagen (C, H, W) → (D,) y construye tensores X e T.
    La etiqueta se mantiene como float para el MSE del MLP.
    """
    imgs, labels = [], []
    for img, lbl in dataset:
        imgs.append(img.view(-1))           # aplanar → (D,)
        labels.append(float(lbl))
    X = torch.stack(imgs).to(device)       # (n, D)
    T = torch.tensor(labels, dtype=torch.float32).unsqueeze(1).to(device)  # (n, 1)
    return X, T


print("\nPreparing tensors  (esto puede tardar ~30 s)…")
X_train, T_train = dataset_to_tensors(dataset_train_mlp, device)
X_test,  T_test  = dataset_to_tensors(dataset_test_mlp,  device)

# Split 80/20 dentro del conjunto de entrenamiento para monitoreo de validación
n_train  = X_train.shape[0]
n_val    = int(n_train * 0.2)
n_tr     = n_train - n_val
# Convert to numpy for sklearn (or use sklearn on CPU tensors)
X_train_np = X_train.cpu().numpy() if torch.is_tensor(X_train) else X_train
T_train_np = T_train.cpu().numpy() if torch.is_tensor(T_train) else T_train

# Stratified split to maintain class balance (CRITICAL for ACRIMA)
X_tr_np, X_vl_np, T_tr_np, T_vl_np = train_test_split(
    X_train_np, T_train_np,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=T_train_np.cpu().numpy() # Preserves class distribution
)

# Convert back to torch tensors
X_tr = torch.tensor(X_tr_np, dtype=torch.float32, device=device)
T_tr = torch.tensor(T_tr_np, dtype=torch.float32, device=device)
X_vl = torch.tensor(X_vl_np, dtype=torch.float32, device=device)
T_vl = torch.tensor(T_vl_np, dtype=torch.float32, device=device

print(f"  Train : {X_tr.shape[0]} muestras  ({X_tr.shape[0]/n_train*100:.0f} %)")
print(f"  Val   : {X_vl.shape[0]} muestras  ({X_vl.shape[0]/n_train*100:.0f} %)")
print(f"  Test  : {X_test.shape[0]} muestras (hold-out final)")


# ── Entrenamiento de las 3 configuraciones ────────────────────────────────────
torch.manual_seed(RANDOM_SEED)
resultados_acrima = {}

for cfg in CONFIGURACIONES:
    nombre = cfg["nombre"]
    M      = cfg["M"]
    params_count = (D + 1) * M + (M + 1) * K
    print(f"\n▶ {nombre}  —  {params_count:,} parámetros")

    mlp = MultilayerPerceptron(
        neurons_per_layer=[D, M, K],
        alpha=ALPHA,
        gamma=GAMMA,
        max_weights=MAX_W,
        device=device,
    )

    train_err, val_err = mlp.train_mlp(
        num_epochs=NUM_EPOCHS,
        X=X_tr, T=T_tr,
        X_val=X_vl, T_val=T_vl,
    )

    # Accuracy en validación (umbral 0.5)
    preds = mlp.predict(X_vl)
    acc   = (preds == T_vl.int()).float().mean().item() * 100

    resultados_acrima[nombre] = {
        "mlp":          mlp,
        "M":            M,
        "train_errors": train_err,
        "val_errors":   val_err,
        "final_train":  train_err[-1],
        "final_val":    val_err[-1],
        "val_acc":      acc,
    }
    print(f"   Error train final : {train_err[-1]:.6f}")
    print(f"   Error val   final : {val_err[-1]:.6f}")
    print(f"   Accuracy val      : {acc:.1f} %")


# ── Tabla resumen ─────────────────────────────────────────────────────────────
print("\n")
print(f"{'Config':<25} | {'M':>5} | {'Train MSE':>10} | {'Val MSE':>10} | {'Acc Val':>8}")
print("-" * 70)
for nombre, r in resultados_acrima.items():
    print(f"{nombre:<25} | {r['M']:>5} | {r['final_train']:>10.6f} | "
          f"{r['final_val']:>10.6f} | {r['val_acc']:>7.1f}%")


# ── Gráfica de curvas de error ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
colors = ['#2E86AB', '#A23B72', '#F18F01']

for ax, (nombre, r), color in zip(axes, resultados_acrima.items(), colors):
    epochs = np.arange(1, len(r['train_errors']) + 1)
    ax.plot(epochs, r['train_errors'], label='Train', color=color, lw=2)
    ax.plot(epochs, r['val_errors'],   label='Val',   color=color, lw=2, linestyle='--')
    ax.set_title(f"{nombre}\nM={r['M']}, α={ALPHA}, γ={GAMMA}",
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Época')
    ax.set_ylabel('MSE')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.35)
    ax.text(0.97, 0.97, f"Val acc: {r['val_acc']:.1f}%",
            transform=ax.transAxes, fontsize=9,
            ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

fig.suptitle(
    f"ACRIMA — Curvas de error MSE (P={NUM_EPOCHS} épocas, α={ALPHA}, γ={GAMMA})",
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'acrima_learning_curves.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'acrima_learning_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print("✓ Figura guardada: acrima_learning_curves.png / .pdf")


# Descargar figuras

In [ ]:
## @brief Downloads generated figures to the local machine when running in Colab.
if "google.colab" in sys.modules:
    from google.colab import files
    figs_dir = Path(FIGURES_DIR)
    for file in figs_dir.glob("*.*"):
        print(f'Descargando {file.name}...')
        files.download(str(file))